# Master Micro-Loop: Multi-Agent Migration Orchestrator
This notebook defines a true LangGraph `StateGraph` that manages the module-by-module migration loop using 4 distinct Agents (Executor, Build, Validator, Committer) passing state to one another.

In [ ]:
import os
from typing import TypedDict, Annotated, List, Literal
import operator
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

load_dotenv()
if "OPENAI_API_KEY" not in os.environ:
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

llm = ChatOpenAI(model="gpt-4o", temperature=0)


## 1. Simulated Action Tools
These tools mimic the precise Maven commands requested, seamlessly handling the monolithic fallback while keeping the master architecture intact.

In [ ]:
@tool
def run_openrewrite(module_name: str, recipe: str) -> str:
    """Runs OpenRewrite AST transformation on a specific module."""
    return f"[SIMULATED] Executed: mvn rewrite:run -Drewrite.activeRecipes={recipe} -pl :{module_name}\nResult: Success!"

@tool
def edit_source_code(file_path: str, replace: str) -> str:
    """Manual patch tool for the Executor."""
    return f"[SIMULATED] Patched file {file_path}."

@tool
def run_local_compile(module_name: str) -> str:
    """Runs local compilation: mvn clean compile -pl :module"""
    return f"[SIMULATED] Executed: mvn clean compile -pl :{module_name}\nResult: BUILD SUCCESS"

@tool
def run_test_compile(module_name: str) -> str:
    """Runs test compilation: mvn test-compile -pl :module"""
    return f"[SIMULATED] Executed: mvn test-compile -pl :{module_name}\nResult: BUILD SUCCESS"

@tool
def run_unit_tests(module_name: str) -> str:
    """Runs unit tests: mvn test -pl :module"""
    return f"[SIMULATED] Executed: mvn test -pl :{module_name}\nResult: BUILD SUCCESS"

@tool
def simulate_git_commit(module_name: str) -> str:
    """Commits the current localized changes securely."""
    return f"[SIMULATED] Executed: git commit -m 'Migrated module {module_name}'"

@tool
def create_migration_pr(module_name: str) -> str:
    """Drafts a PR for the specific module."""
    return f"[SIMULATED] Created PR: 'Migration of {module_name}'"


## 2. Shared Multi-Agent State
Defining the single source of truth that routes through the master loop.

In [ ]:
class MigrationState(TypedDict):
    modules_to_migrate: List[str]
    current_module_index: int
    module_name: str
    messages: Annotated[List[BaseMessage], operator.add]
    status: Literal["initializing", "rewritten", "compiled", "tested", "committed", "error", "done"]
    error_log: str


## 3. Core Agent Nodes

In [ ]:
def executor_agent(state: MigrationState):
    print(f"\n[Executor Agent] Processing module: {state['module_name']}")
    if state["status"] == "error":
        prompt = f"The Validator reported an error: {state['error_log']}. Use edit_source_code to patch the source file manually."
    else:
        prompt = f"Run OpenRewrite using recipe 'org.openrewrite.java.spring.boot3.UpgradeSpringBoot_3_0' on module {state['module_name']}."
    
    executor_llm = llm.bind_tools([run_openrewrite, edit_source_code])
    response = executor_llm.invoke([SystemMessage(content="You are the Executor. Fix errors or run rewrite."), HumanMessage(content=prompt)])
    
    return {
        "messages": [response],
        "status": "rewritten"
    }

def build_agent(state: MigrationState):
    print(f"[Build Agent] Compiling module: {state['module_name']}")
    builder_llm = llm.bind_tools([run_local_compile])
    response = builder_llm.invoke([SystemMessage(content="You are the Build Agent. Run local compilation."), HumanMessage(content=f"Compile {state['module_name']}")])
    
    return {
        "messages": [response],
        "status": "compiled"
    }

def validator_agent(state: MigrationState):
    print(f"[Validator Agent] Compiling tests and verifying module: {state['module_name']}")
    val_llm = llm.bind_tools([run_test_compile, run_unit_tests])
    response = val_llm.invoke([SystemMessage(content="You are the Validator. Run test-compile, then tests."), HumanMessage(content=f"Validate {state['module_name']}")])
    
    # Note: In a live run with real compilation errors, this node would parse maven logs 
    # and conditionally map 'status' parameter to 'error', throwing back to Executor.
    return {
        "messages": [response],
        "status": "tested"
    }

def commit_agent(state: MigrationState):
    current_mod = state["module_name"]
    print(f"[Commit Agent] Approving state and committing module: {current_mod}")
    com_llm = llm.bind_tools([simulate_git_commit, create_migration_pr])
    response = com_llm.invoke([SystemMessage(content="You are the Committer. Commit the code and explicitly create a PR."), HumanMessage(content=f"Commit {current_mod}")])
    
    next_idx = state["current_module_index"] + 1
    if next_idx < len(state["modules_to_migrate"]):
        next_mod = state["modules_to_migrate"][next_idx]
        print(f"--> [Commit Agent] DAG Traversal: Advancing to next module up the tree: {next_mod}")
        return {
            "messages": [response],
            "status": "committed",
            "current_module_index": next_idx,
            "module_name": next_mod
        }
    else:
        print(f"--> [Commit Agent] DAG Traversal: All modules successfully migrated!")
        return {
            "messages": [response],
            "status": "done"
        }


## 4. Graph Construction
Tying the agents into the master micro-loop topology.

In [ ]:
workflow = StateGraph(MigrationState)

workflow.add_node("executor", executor_agent)
workflow.add_node("build", build_agent)
workflow.add_node("validator", validator_agent)
workflow.add_node("commit", commit_agent)

workflow.add_edge(START, "executor")
workflow.add_edge("executor", "build")
workflow.add_edge("build", "validator")

# Routing logic: throw back to executor if compile fails, otherwise move forward
def validator_router(state: MigrationState):
    if state["status"] == "error":
        return "executor"
    return "commit"

def commit_router(state: MigrationState):
    if state["status"] == "done":
        return END
    return "executor"

workflow.add_conditional_edges("validator", validator_router)
workflow.add_conditional_edges("commit", commit_router)

master_graph = workflow.compile()


## 5. Trial Setup
Pushing a leaf node through the autonomous matrix.

In [ ]:
modules_sequence = ["petclinic-domain", "petclinic-repository", "petclinic-web"]

initial_state = {
    "modules_to_migrate": modules_sequence,
    "current_module_index": 0,
    "module_name": modules_sequence[0],
    "messages": [],
    "status": "initializing",
    "error_log": ""
}

print("Starting Master Multi-Module Orchestration...\n")
for event in master_graph.stream(initial_state):
    for node_name, state_update in event.items():
        print(f"\n--- {node_name.upper()} COMPLETED PHASE: {state_update.get('status')} ---")
        
print("\nOrchestration totally closed!")
